# HW04 作业：序列模型、循环神经网络、嵌入向量与注意力机制

<div style="padding:10px 14px;border-left:5px solid #2E7D32;background:#F6FFF6;line-height:1.8;">
<b>课程：</b> 深度学习<br>
<b>姓名：</b> 刘佳雨<br>
<b>学号：</b> 20234080114<br>
<b>说明：</b> 本 notebook 按 HW04 题目顺序完成理论计算题与编程题。编程部分均打印样例输出，并加入形状检查、概率归一化检查或梯度检查，保证结果可复现、可验证。
</div>

## 目录
- 2 序列模型
- 3 循环神经网络
- 4 高级循环神经网络
- 5 嵌入向量
- 6 注意力机制


## 2 序列模型

### 2.1 理论计算题

给定字符序列 `"ababc"`，一阶马尔可夫模型只统计相邻字符转移：

$$a\to b,\quad b\to a,\quad a\to b,\quad b\to c$$

词汇表为 $\{a,b,c\}$，大小 $|V|=3$。对给定前一个字符 $x_{t-1}=b$，从 `b` 出发的转移共有 2 次：

$$C(b,a)=1,\quad C(b,b)=0,\quad C(b,c)=1,\quad C(b,*)=2$$

使用拉普拉斯平滑：

$$p(x_t=y\mid x_{t-1}=b)=\frac{C(b,y)+1}{C(b,*)+|V|}$$

因此：

$$p(a\mid b)=\frac{1+1}{2+3}=\frac{2}{5}=0.4$$

$$p(c\mid b)=\frac{1+1}{2+3}=\frac{2}{5}=0.4$$

顺带可检查三个条件概率之和：

$$p(a\mid b)+p(b\mid b)+p(c\mid b)=\frac{2}{5}+\frac{1}{5}+\frac{2}{5}=1$$


### 2.2 编程题


In [1]:
import re
from collections import Counter


def preprocess_text(text, n):
    '''
    文本预处理并构造自回归语言模型样本。
    - 小写化；
    - 去除标点，只保留英文字母和空格；
    - 空格分词；
    - 按词频从高到低构建词汇表，ID 从 0 开始；
    - 若词频相同，按首次出现顺序排序，保证结果可复现；
    - 用长度为 n 的滑动窗口生成特征；若没有下一个词，则标签记为 None，
      与题目给出的示例保持一致。
    '''
    clean = re.sub(r"[^a-zA-Z\s]", " ", text.lower())
    tokens = clean.split()
    counts = Counter(tokens)
    first_pos = {word: tokens.index(word) for word in counts}
    sorted_words = sorted(counts, key=lambda w: (-counts[w], first_pos[w]))
    vocab = {word: idx for idx, word in enumerate(sorted_words)}

    features, labels = [], []
    for i in range(max(0, len(tokens) - n + 1)):
        window = tokens[i : i + n]
        next_index = i + n
        label = tokens[next_index] if next_index < len(tokens) else None
        features.append(window)
        labels.append(label)
    feature_ids = [[vocab[word] for word in window] for window in features]
    label_ids = [vocab[label] if label is not None else None for label in labels]
    return vocab, (features, labels), (feature_ids, label_ids)


text = "The time machine, the Time traveler!"
vocab, (features, labels), (feature_ids, label_ids) = preprocess_text(text, n=2)
print("词汇表:", vocab)
print("特征:", features)
print("标签:", labels)
print("特征 ID:", feature_ids)
print("标签 ID:", label_ids)

vocab_ex, (features_ex, labels_ex), _ = preprocess_text("The time machine", n=2)
print("\n题目示例特征:", features_ex)
print("题目示例标签:", labels_ex)
assert features_ex == [["the", "time"], ["time", "machine"]]
assert labels_ex == ["machine", None]


词汇表: {'the': 0, 'time': 1, 'machine': 2, 'traveler': 3}
特征: [['the', 'time'], ['time', 'machine'], ['machine', 'the'], ['the', 'time'], ['time', 'traveler']]
标签: ['machine', 'the', 'time', 'traveler', None]
特征 ID: [[0, 1], [1, 2], [2, 0], [0, 1], [1, 3]]
标签 ID: [2, 0, 1, 3, None]

题目示例特征: [['the', 'time'], ['time', 'machine']]
题目示例标签: ['machine', None]


## 3 循环神经网络

### 3.1 理论计算题

线性 RNN 定义为：

$$h_t=W_{hh}h_{t-1}+W_{hx}x_t,\qquad o_t=W_{oh}h_t$$

平方损失为：

$$L=\frac{1}{2}\sum_{t=1}^{T}(o_t-y_t)^2$$

令输出误差为：

$$e_t=\frac{\partial L}{\partial o_t}=o_t-y_t$$

因为 $h_t$ 会影响当前输出以及未来所有隐藏状态，所以通过时间反向传播有：

$$\delta_t=\frac{\partial L}{\partial h_t}=W_{oh}^{T}e_t+W_{hh}^{T}\delta_{t+1},\qquad \delta_{T+1}=0$$

这里 $\delta_t$ 同时包含当前时刻输出误差带来的梯度，以及未来时刻经由隐藏状态递归传回来的梯度。

将递推式完全展开：

$$\delta_t=\sum_{k=t}^{T}(W_{hh}^{T})^{k-t}W_{oh}^{T}e_k$$

而

$$\frac{\partial h_t}{\partial W_{hh}} \Rightarrow \frac{\partial L}{\partial W_{hh}}
=\sum_{t=1}^{T}\delta_t h_{t-1}^{T}$$

其中 $\delta_t h_{t-1}^{T}$ 是外积，形状与 $W_{hh}$ 相同。

代入展开式，可得：

$$
\frac{\partial L}{\partial W_{hh}}
=\sum_{t=1}^{T}\left[\sum_{k=t}^{T}(W_{hh}^{T})^{k-t}W_{oh}^{T}(o_k-y_k)\right]h_{t-1}^{T}
$$

梯度消失或爆炸主要由反复相乘的 $(W_{hh}^{T})^{k-t}$ 决定。若 $W_{hh}$ 的谱半径 $\rho(W_{hh})<1$，长时间步梯度趋向 0，容易梯度消失；若 $\rho(W_{hh})>1$，梯度范数可能指数增长，容易梯度爆炸。当 $\rho(W_{hh})$ 接近 1 且矩阵条件较好时，梯度传播相对稳定。


### 3.2 编程题


In [2]:
import numpy as np


def rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h):
    '''单步 RNN 前向传播，激活函数为 tanh。'''
    a_t = x_t @ W_hx + h_prev @ W_hh + b_h
    h_t = np.tanh(a_t)
    cache = (x_t, h_prev, W_hx, W_hh, a_t, h_t)
    return h_t, cache


def rnn_cell_backward(dh_next, cache):
    '''单步 RNN 反向传播，只计算梯度，不更新参数。'''
    x_t, h_prev, W_hx, W_hh, a_t, h_t = cache
    da = dh_next * (1 - h_t ** 2)
    dx_t = da @ W_hx.T
    dh_prev = da @ W_hh.T
    dW_hx = x_t.T @ da
    dW_hh = h_prev.T @ da
    db_h = da.sum(axis=0)
    return dx_t, dh_prev, dW_hx, dW_hh, db_h


np.random.seed(0)
batch_size, input_size, hidden_size = 2, 3, 4
x_t = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hx = np.random.randn(input_size, hidden_size) * 0.1
W_hh = np.random.randn(hidden_size, hidden_size) * 0.1
b_h = np.zeros(hidden_size)
dh_next = np.random.randn(batch_size, hidden_size)

h_t, cache = rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h)
dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_cell_backward(dh_next, cache)

print("h_t shape:", h_t.shape)
print("dx_t shape:", dx_t.shape)
print("dh_prev shape:", dh_prev.shape)
print("dW_hx shape:", dW_hx.shape)
print("dW_hh shape:", dW_hh.shape)
print("db_h shape:", db_h.shape)
print("\nh_t:\n", np.round(h_t, 4))
print("\ndb_h:\n", np.round(db_h, 4))

# 用有限差分检查 dW_hh。令局部标量目标 J = sum(h_t * dh_next)，
# 则反向传播得到的 dW_hh 应当等于 dJ/dW_hh。
def local_objective(W_hh_value):
    h, _ = rnn_cell_forward(x_t, h_prev, W_hx, W_hh_value, b_h)
    return np.sum(h * dh_next)


eps = 1e-5
num_dW_hh = np.zeros_like(W_hh)
for i in range(W_hh.shape[0]):
    for j in range(W_hh.shape[1]):
        old = W_hh[i, j]
        W_hh[i, j] = old + eps
        loss_plus = local_objective(W_hh)
        W_hh[i, j] = old - eps
        loss_minus = local_objective(W_hh)
        W_hh[i, j] = old
        num_dW_hh[i, j] = (loss_plus - loss_minus) / (2 * eps)

max_error = np.max(np.abs(num_dW_hh - dW_hh))
print("\ndW_hh 有限差分最大误差:", f"{max_error:.2e}")
assert max_error < 1e-8


h_t shape: (2, 4)
dx_t shape: (2, 3)
dh_prev shape: (2, 4)
dW_hx shape: (3, 4)
dW_hh shape: (4, 4)
db_h shape: (4,)

h_t:
 [[ 0.1636 -0.0853  0.4517 -0.0535]
 [ 0.0654  0.0483 -0.3713  0.0256]]

db_h:
 [-2.908   2.7123 -1.797  -0.6494]

dW_hh 有限差分最大误差: 1.27e-10


## 4 高级循环神经网络

### 4.1 理论计算题

设每层每个方向都是普通 RNN。第 1 层每个方向的参数为：

$$W_{xh}:D\times H,\quad W_{hh}:H\times H,\quad b:H$$

所以第 1 层双向参数量为：

$$2(DH+H^2+H)$$

从第 2 层开始，每层输入都是上一层前向和后向隐藏状态拼接，输入维度为 $2H$。因此每个方向参数为：

$$2H\cdot H+H^2+H=3H^2+H$$

其余 $L-1$ 层双向参数量为：

$$2(L-1)(3H^2+H)$$

最后输出层从 $2H$ 映射到 $O$：

$$2HO+O$$

因此总参数量为：

$$
N=2(DH+H^2+H)+2(L-1)(3H^2+H)+2HO+O
$$

若采用 PyTorch RNN 默认的 `bias_ih` 与 `bias_hh` 两组偏置，则每个方向每层偏置从 $H$ 变为 $2H$，相应地把上式循环层中的偏置项加倍即可。


### 4.2 编程题


In [3]:
def bidirectional_rnn_encoder(X, W_xh_f, W_hh_f, b_f, W_xh_b, W_hh_b, b_b):
    '''
    手动实现双向 RNN 编码器。
    X: (seq_len, batch, input_dim)
    返回:
      outputs: (seq_len, batch, 2*hidden_dim)
      final_state: (batch, 2*hidden_dim)，由最后前向状态和最后后向状态拼接
    '''
    seq_len, batch, _ = X.shape
    hidden_dim = b_f.shape[0]

    h_f = np.zeros((seq_len, batch, hidden_dim))
    h_b = np.zeros((seq_len, batch, hidden_dim))

    prev = np.zeros((batch, hidden_dim))
    for t in range(seq_len):
        prev = np.tanh(X[t] @ W_xh_f + prev @ W_hh_f + b_f)
        h_f[t] = prev

    prev = np.zeros((batch, hidden_dim))
    for t in range(seq_len - 1, -1, -1):
        prev = np.tanh(X[t] @ W_xh_b + prev @ W_hh_b + b_b)
        h_b[t] = prev

    outputs = np.concatenate([h_f, h_b], axis=2)
    final_state = np.concatenate([h_f[-1], h_b[0]], axis=1)
    return outputs, final_state


np.random.seed(1)
seq_len, batch, input_dim, hidden_dim = 5, 2, 3, 4
X = np.random.randn(seq_len, batch, input_dim)
W_xh_f = np.random.randn(input_dim, hidden_dim) * 0.1
W_hh_f = np.random.randn(hidden_dim, hidden_dim) * 0.1
b_f = np.zeros(hidden_dim)
W_xh_b = np.random.randn(input_dim, hidden_dim) * 0.1
W_hh_b = np.random.randn(hidden_dim, hidden_dim) * 0.1
b_b = np.zeros(hidden_dim)

outputs, final_state = bidirectional_rnn_encoder(X, W_xh_f, W_hh_f, b_f, W_xh_b, W_hh_b, b_b)
assert outputs.shape == (seq_len, batch, 2 * hidden_dim)
assert final_state.shape == (batch, 2 * hidden_dim)
assert np.allclose(final_state[:, :hidden_dim], outputs[-1, :, :hidden_dim])
assert np.allclose(final_state[:, hidden_dim:], outputs[0, :, hidden_dim:])
print("outputs shape:", outputs.shape)
print("final_state shape:", final_state.shape)
print("第 0 个时间步拼接隐藏状态:\n", np.round(outputs[0], 4))
print("序列表示 final_state:\n", np.round(final_state, 4))
print("检查: final_state = [最后前向隐藏状态; 最后后向隐藏状态] -> 通过")


outputs shape: (5, 2, 8)
final_state shape: (2, 8)
第 0 个时间步拼接隐藏状态:
 [[-0.1576 -0.1025 -0.0331 -0.1044 -0.0434  0.0404 -0.2606  0.2966]
 [-0.3504 -0.1286  0.0212  0.3052 -0.2341 -0.3933 -0.3459  0.1489]]
序列表示 final_state:
 [[-0.0236 -0.0842  0.0289 -0.0587 -0.0434  0.0404 -0.2606  0.2966]
 [ 0.1693  0.0607  0.0953  0.0236 -0.2341 -0.3933 -0.3459  0.1489]]
检查: final_state = [最后前向隐藏状态; 最后后向隐藏状态] -> 通过


## 5 嵌入向量

### 5.1 理论计算题

Skip-gram 的目标是给定中心词 $w_c$，预测上下文词 $w_o$。使用负采样时，对一个正样本 $(w_c,w_o)$，再从噪声分布 $P_n(w)$ 中采样 $K$ 个负样本 $w_{n_1},\ldots,w_{n_K}$。

令中心词向量为 $v_c$，上下文词输出向量为 $u_o$，第 $k$ 个负样本输出向量为 $u_{n_k}$，则最大化的对数似然目标为：

$$
\log \sigma(u_o^T v_c)+\sum_{k=1}^{K}\log \sigma(-u_{n_k}^{T}v_c)
$$

对应最小化损失为：

$$
\mathcal{L}
=-\log \sigma(u_o^T v_c)-\sum_{k=1}^{K}\log \sigma(-u_{n_k}^{T}v_c)
$$

其中：

$$\sigma(x)=\frac{1}{1+e^{-x}}$$

负样本通常从噪声分布中独立采样，常见做法是使用词频的 $3/4$ 次幂平滑：

$$
P_n(w)=\frac{f(w)^{3/4}}{\sum_{w'\in V}f(w')^{3/4}}
$$

采样时应避免把当前正样本上下文词当作负样本；实际实现中也常排除中心词或进行重采样。


### 5.2 编程题


In [4]:
def softmax(logits, axis=-1):
    logits = logits - np.max(logits, axis=axis, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=axis, keepdims=True)


def cbow_forward_loss(context_indices, target_indices, W, W_out):
    '''
    CBOW 完整 softmax 前向传播和交叉熵损失。
    context_indices: (batch, context_size)
    target_indices: (batch,)
    W: (V, d)
    W_out: (d, V)
    '''
    context_emb = W[context_indices]
    hidden = context_emb.mean(axis=1)
    logits = hidden @ W_out
    probs = softmax(logits, axis=1)
    batch_indices = np.arange(target_indices.shape[0])
    loss = -np.log(probs[batch_indices, target_indices] + 1e-12).mean()
    return loss, probs, hidden


np.random.seed(2)
V, d, context_size, batch = 6, 4, 3, 2
W = np.random.randn(V, d) * 0.1
W_out = np.random.randn(d, V) * 0.1
context_indices = np.array([[0, 1, 2], [2, 3, 4]])
target_indices = np.array([3, 5])

loss, probs, hidden = cbow_forward_loss(context_indices, target_indices, W, W_out)
manual_loss = -np.mean([np.log(probs[i, target_indices[i]] + 1e-12) for i in range(batch)])
assert hidden.shape == (batch, d)
assert probs.shape == (batch, V)
assert np.allclose(probs.sum(axis=1), 1.0)
assert np.allclose(loss, manual_loss)
print("hidden shape:", hidden.shape)
print("probs shape:", probs.shape)
print("loss:", round(float(loss), 6))
print("每个样本的概率分布:\n", np.round(probs, 4))
print("每行概率和:", np.round(probs.sum(axis=1), 6))
print("手工交叉熵校验:", round(float(manual_loss), 6))


hidden shape: (2, 4)
probs shape: (2, 6)
loss: 1.794906
每个样本的概率分布:
 [[0.1632 0.1666 0.1712 0.1665 0.1686 0.1639]
 [0.1648 0.1667 0.169  0.1665 0.1673 0.1657]]
每行概率和: [1. 1.]
手工交叉熵校验: 1.794906


## 6 注意力机制

### 6.1 理论计算题

题目给出矩阵形状：

$$Q\in\mathbb{R}^{2\times4},\quad K\in\mathbb{R}^{3\times4},\quad V\in\mathbb{R}^{3\times5},\quad d_k=4$$

由于题目未给出具体矩阵元素，这里写出完整的符号计算过程。设：

$$
Q=\begin{bmatrix}q_1\\q_2\end{bmatrix},\quad
K=\begin{bmatrix}k_1\\k_2\\k_3\end{bmatrix},\quad
V=\begin{bmatrix}v_1\\v_2\\v_3\end{bmatrix}
$$

其中 $q_i,k_j\in\mathbb{R}^{4}$，$v_j\in\mathbb{R}^{5}$。

第一步，计算缩放得分矩阵：

$$
S=\frac{QK^T}{\sqrt{d_k}}=\frac{QK^T}{2}
=
\begin{bmatrix}
\frac{q_1\cdot k_1}{2} & \frac{q_1\cdot k_2}{2} & \frac{q_1\cdot k_3}{2}\\
\frac{q_2\cdot k_1}{2} & \frac{q_2\cdot k_2}{2} & \frac{q_2\cdot k_3}{2}
\end{bmatrix}
\in\mathbb{R}^{2\times3}
$$

第二步，对每一行做 softmax：

$$
A_{ij}=\frac{\exp(S_{ij})}{\sum_{m=1}^{3}\exp(S_{im})},\qquad A\in\mathbb{R}^{2\times3}
$$

第三步，加权求和值向量：

$$
O=AV
=
\begin{bmatrix}
A_{11}v_1+A_{12}v_2+A_{13}v_3\\
A_{21}v_1+A_{22}v_2+A_{23}v_3
\end{bmatrix}
\in\mathbb{R}^{2\times5}
$$

这就是无掩码缩放点积注意力的输出矩阵。


#### 6.1 数值计算示例


In [5]:
Q_demo = np.array([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 1.0, 0.0, 1.0],
])
K_demo = np.array([
    [1.0, 0.0, 0.0, 0.0],
    [0.0, 1.0, 0.0, 0.0],
    [1.0, 1.0, 0.0, 0.0],
])
V_demo = np.array([
    [1.0, 0.0, 0.0, 0.0, 1.0],
    [0.0, 1.0, 0.0, 1.0, 0.0],
    [1.0, 1.0, 1.0, 0.0, 0.0],
])

scores_demo = Q_demo @ K_demo.T / np.sqrt(4)
weights_demo = softmax(scores_demo, axis=1)
output_demo = weights_demo @ V_demo

assert scores_demo.shape == (2, 3)
assert weights_demo.shape == (2, 3)
assert output_demo.shape == (2, 5)
assert np.allclose(weights_demo.sum(axis=1), 1.0)

print("QK^T / sqrt(d_k) 得分矩阵:\n", np.round(scores_demo, 4))
print("\nsoftmax 注意力权重:\n", np.round(weights_demo, 4))
print("\n注意力输出矩阵:\n", np.round(output_demo, 4))
print("\n每行注意力权重之和:", np.round(weights_demo.sum(axis=1), 6))


QK^T / sqrt(d_k) 得分矩阵:
 [[0.5 0.  0.5]
 [0.  0.5 0.5]]

softmax 注意力权重:
 [[0.3837 0.2327 0.3837]
 [0.2327 0.3837 0.3837]]

注意力输出矩阵:
 [[0.7673 0.6163 0.3837 0.2327 0.3837]
 [0.6163 0.7673 0.3837 0.3837 0.2327]]

每行注意力权重之和: [1. 1.]


### 6.2 编程题


In [6]:
def multi_head_attention_forward(X, W_q, W_k, W_v, W_o, b_q=None, b_k=None, b_v=None, b_o=None, num_heads=2):
    '''
    NumPy 实现多头注意力前向传播。
    X: (seq_len, batch, d_model)
    W_q, W_k, W_v, W_o: (d_model, d_model)
    返回 output: (seq_len, batch, d_model)
    '''
    seq_len, batch, d_model = X.shape
    assert d_model % num_heads == 0
    d_k = d_model // num_heads

    b_q = 0 if b_q is None else b_q
    b_k = 0 if b_k is None else b_k
    b_v = 0 if b_v is None else b_v
    b_o = 0 if b_o is None else b_o

    Q = X @ W_q + b_q
    K = X @ W_k + b_k
    V = X @ W_v + b_v

    def split_heads(T):
        # (seq_len, batch, d_model) -> (batch, num_heads, seq_len, d_k)
        return T.reshape(seq_len, batch, num_heads, d_k).transpose(1, 2, 0, 3)

    Qh, Kh, Vh = split_heads(Q), split_heads(K), split_heads(V)
    scores = Qh @ Kh.transpose(0, 1, 3, 2) / np.sqrt(d_k)
    attn = softmax(scores, axis=-1)
    context = attn @ Vh

    # (batch, num_heads, seq_len, d_k) -> (seq_len, batch, d_model)
    concat = context.transpose(2, 0, 1, 3).reshape(seq_len, batch, d_model)
    output = concat @ W_o + b_o
    return output, attn


np.random.seed(3)
seq_len, batch, d_model, num_heads = 4, 2, 4, 2
X = np.random.randn(seq_len, batch, d_model)
W_q = np.random.randn(d_model, d_model) * 0.1
W_k = np.random.randn(d_model, d_model) * 0.1
W_v = np.random.randn(d_model, d_model) * 0.1
W_o = np.random.randn(d_model, d_model) * 0.1

output, attn = multi_head_attention_forward(X, W_q, W_k, W_v, W_o, num_heads=num_heads)
assert output.shape == X.shape
assert attn.shape == (batch, num_heads, seq_len, seq_len)
assert np.allclose(attn.sum(axis=-1), 1.0)
print("output shape:", output.shape)
print("attention shape:", attn.shape)
print("output[0]:\n", np.round(output[0], 4))
print("每个注意力分布的行和是否为 1:\n", np.round(attn.sum(axis=-1), 6))
print("第 1 个 batch、第 1 个 head 的注意力矩阵:\n", np.round(attn[0, 0], 4))


output shape: (4, 2, 4)
attention shape: (2, 2, 4, 4)
output[0]:
 [[-0.0249 -0.0344 -0.0054  0.0084]
 [-0.0041  0.0274 -0.0027 -0.0158]]
每个注意力分布的行和是否为 1:
 [[[1. 1. 1. 1.]
  [1. 1. 1. 1.]]

 [[1. 1. 1. 1.]
  [1. 1. 1. 1.]]]
第 1 个 batch、第 1 个 head 的注意力矩阵:
 [[0.2633 0.2447 0.251  0.241 ]
 [0.2363 0.2451 0.2565 0.2621]
 [0.2351 0.2787 0.2322 0.254 ]
 [0.239  0.2669 0.2398 0.2543]]
